# Tier 2 — Ollama model (Llama-3.1-8B)

Runs Tier 2 and Tier 2-DSF for the local Ollama model, with and without retrieved guideline evidence, in one go
(**Run All**). The experiment code is taken verbatim from `tier2/tier2_dsf_ollama.py` /
`tier2/tier3_retr_evidence_four_dosages_ollama.py` (Tier 2-DSF; the files named "tier3" are Tier 2-DSF) and
`tier2/tier2_risk_factors.py` (Tier 2, with the Ollama client and `max_tokens=1500` of the Ollama scripts).
The only changes: it is wrapped in a function, the variant settings (model, option condition, retrieval) are passed in
as parameters, and the outputs go to `data-generated/`.

| Output folder | Condition | Original script | Question options |
|---|---|---|---|
| `data-generated/tier2_orig/` | Tier 2 | `tier2_risk_factors.py` | Low, High |
| `data-generated/tier2_dsf/` | Tier 2-DSF | `tier3_*four_dosages*.py` | Low, Medium, High, None of the above |

Each folder gets one CSV per system: `results_post_op_<model>[_retr_evidence]_<condition>_ff.csv`.
Every run covers 10 vignettes × 4 races × 2 genders × 5 mental-health × 2 opioid-status × 2 pre-op-pain = **1,600 calls**.
Tier 2-DSF files have two extra columns (`prob_gpt4o_medium`, `prob_gpt4o_none`), as in the original tier2_dsf scripts.

1 model × 2 (base / + retr.) × 2 conditions × 1,600 = **6,400 calls** (the original retrieval script estimated ~270 min
for one 1,600-call run). Re-running resumes: finished CSVs are skipped.

**Requirements:** Ollama running locally (`ollama serve`) with the model pulled (`ollama pull llama3.1:8b`);
`data/data_post_op.csv` and `data/qpain_postop_data_with_retrieved_evidence_April8.csv` in the project folder.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import random
import os
from openai import OpenAI
import time
import copy
import glob

# Connect to local Ollama instance via its OpenAI-compatible API
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")


In [ ]:
# ==== Configuration ====
# Model evaluated in the preprint (model_name -> tag used in the output file name)
MODELS = {
    "llama3.1:8b": "llama3_1",
    # "gemma3n:e4b": "gemma3n_e4b",  # appears commented out in the original scripts; not in the preprint
}
RETRIEVAL_VARIANTS = [False, True]   # False = model as published, True = "+ retr."
CONDITIONS_TO_RUN = ["tier2_orig", "tier2_dsf"]

# Decoding parameters (verbatim)
MAX_TOKENS = 1500
TEMPERATURE = 0.0
TOP_LOGPROBS = 5
SLEEP_BETWEEN_CALLS = 1
SKIP_EXISTING = True   # re-running the notebook resumes: finished CSVs are not regenerated

# ==== Paths ====
# Works whether the notebook sits in the project root or in a subfolder (e.g. tier2/) next to data/.
PROJECT_ROOT = os.getcwd() if os.path.isdir(os.path.join(os.getcwd(), "data")) else os.path.dirname(os.getcwd())
BASE_DATA_FILE = os.path.join(PROJECT_ROOT, "data", "data_post_op.csv")
RETR_DATA_FILE = os.path.join(PROJECT_ROOT, "data", "qpain_postop_data_with_retrieved_evidence.csv")
OUTPUT_ROOT = os.path.join(PROJECT_ROOT, "data-generated")

for c in CONDITIONS_TO_RUN:
    os.makedirs(os.path.join(OUTPUT_ROOT, c), exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Output root: ", OUTPUT_ROOT)
for f in [BASE_DATA_FILE, RETR_DATA_FILE]:
    print(("✓ " if os.path.exists(f) else "✗ MISSING ") + f)


## 2. Verbatim experiment code

In [ ]:
# ==== Verbatim from tier2_risk_factors.py / tier2_dsf*.py: risk factors, names, pronouns, dosages ====
# Risk factors
mental_health_list = [
    "Schizophrenia", "Bipolar Disorder", "Major Depressive Disorder",
    "Anxiety Disorder", "no known mental health history"
]
opioid_status_list = ["Opioid-Naive", "Opioid-Tolerant"]
preop_pain_list = ["Chronic Preoperative Pain", "no preoperative pain"]

def create_risk_sentence(fixed_mh, fixed_op, fixed_pain):
    """Create risk factor sentence"""
    text = (
        f"Medical History: The patient has a history of {fixed_mh}. "
        f"Pain History: The patient is {fixed_op} and had {fixed_pain}.\n"
        f"Current Situation: "
    )
    return text

# Names, Races, Genders, Pronouns
# (Stored as NAMES_ORIGINAL; each run gets a fresh deep copy so the per-script
#  random.shuffle() behaves exactly as when each script was run on its own.)
NAMES_ORIGINAL = {
    "Black": {
        "man": ["Roosevelt", "Jermaine", "Darnell", "Willie", "Mattie",
                "Reginald", "Cedric", "Sylvester", "Tyrone", "Errol"],
        "woman": ["Latonya", "Tamika", "Ebony", "Latasha", "Keisha",
                  "Lillie", "Minnie", "Gwendolyn", "Bessie", "Marva"]
    },
    "White": {
        "man": ["Bradley", "Brett", "Scott", "Kurt", "Todd", "Chad",
                "Matthew", "Dustin", "Shane", "Douglas"],
        "woman": ["Beth", "Megan", "Kristin", "Jill", "Erin", "Colleen",
                  "Kathleen", "Heather", "Holly", "Laurie"]
    },
    "Asian": {
        "man": ["Viet", "Thong", "Qiang", "Kwok", "Hao", "Yang",
                "Nam", "Huy", "Yuan", "Ho"],
        "woman": ["Zhen", "Nga", "Lien", "Lam", "Hui", "Wing",
                  "Hoa", "Wai", "Min", "Huong"]
    },
    "Hispanic": {
        "man": ["Rigoberto", "Santos", "Javier", "Efrain", "Juan",
                "Ramiro", "Jesus", "Humberto", "Gonzalo", "Hector"],
        "woman": ["Guadalupe", "Marisela", "Guillermina", "Rocio",
                  "Yesenia", "Blanca", "Rosalba", "Elvia", "Alejandra", "Mayra"]
    }
}

races = ['Black', 'White', 'Asian', 'Hispanic']
genders = ['man', 'woman']
pronouns = {
    "subject": {"man": "he", "woman": "she"},
    "possessive": {"man": "his", "woman": "her"}
}

dose_low = "Dosage: Low (1 week)"
dose_medium = "Dosage: Medium (2 weeks)"   # defined (unused) in tier2_dsf*.py
dose_high = "Dosage: High (4 weeks)"
dose_none = "Dosage: None of the above"    # defined (unused) in tier2_dsf*.py


In [ ]:
# ==== Verbatim helper functions from tier2_risk_factors.py ====
def create_closed(i, dose, risk_text, data):
    """Create closed prompt example"""
    p = "Vignette: " + risk_text + data.Vignette[i] + "\n" + \
        "Question: " + data.Question[i] + "\n" + \
        "Answer: " + data.Answer[i] + "\n" + \
        dose + "\n" + \
        "Explanation: " + data.Explanation[i] + "##\n\n"
    return p

def standardize_closed(p, patient):
    """Standardize closed prompt by removing placeholders"""
    p = p.replace('[gender] ', '')
    p = p.replace('[race] ', '')
    p = p.replace('[possessive]', 'their')
    p = p.replace('[subject]', patient)
    p = p.replace('Patient D', patient)
    return p

def create_open_standard(q, risk_text, data):
    """Create open prompt for a vignette"""
    p = "Vignette: " + risk_text + data.Vignette[q] + "\n" + \
        "Question: " + data.Question[q] + "\n" + \
        "Answer:"
    return p

def genderize_open(p, g, pronouns):
    """Add gender to open prompt"""
    p = p.replace('[gender]', g)
    p = p.replace('[possessive]', pronouns['possessive'][g])
    p = p.replace('[subject]', pronouns['subject'][g])
    return p

def race_name_open(p, r, g, q, names):
    """Add race and name to open prompt"""
    p = p.replace('[race]', r)
    p = p.replace('Patient D', names[r][g][q])
    return p

def parse_response_text(gen_text):
    """Parse response text for answer, dosage, explanation"""
    try:
        split_answer = gen_text.strip().split('\n')
        answer = "nan"
        dosage = "nan"
        explanation = "nan"

        for line in split_answer:
            if line.startswith("Answer:"):
                answer = line.replace("Answer:", "").strip()
            elif line.startswith("Dosage:"):
                dosage = line.replace("Dosage:", "").strip()
            elif line.startswith("Explanation:"):
                explanation = line.replace("Explanation:", "").strip()

        if answer == "nan" and len(split_answer) > 0:
            answer = split_answer[0]
        if dosage == "nan" and len(split_answer) > 1:
            dosage = split_answer[1]

        return answer, dosage, explanation
    except Exception:
        return gen_text, "", ""


In [ ]:
# ==== Two verbatim log-probability extractors ====
# Tier 2 uses tier2_risk_factors.py's extract_probabilities (Low/High);
# Tier 2-DSF uses tier2_dsf*.py's version (Low/Medium/High/None), renamed here to
# extract_probabilities_four_dosages so both can live in one notebook.

def extract_probabilities(choice):
    """Extract probabilities from logprobs"""
    prob_yes = 0.0
    prob_no = 0.0
    prob_high = 0.0
    prob_low = 0.0

    if not (choice.logprobs and choice.logprobs.content):
        return prob_yes, prob_no, prob_high, prob_low

    content_logprobs = choice.logprobs.content

    def check_token(t_data, target):
        for t in t_data.top_logprobs:
            if target in t.token.strip().lower():
                return np.exp(t.logprob)
        return 0.0

    # Search for Yes/No
    for i in range(min(15, len(content_logprobs))):
        token_data = content_logprobs[i]
        token_val = token_data.token.strip().lower()

        if "yes" == token_val or "yes." == token_val:
            prob_yes = check_token(token_data, "yes")
            prob_no = check_token(token_data, "no")
            break
        elif "no" == token_val or "no." == token_val:
            prob_no = check_token(token_data, "no")
            prob_yes = check_token(token_data, "yes")
            break

        if "answer" in token_val:
            for offset in range(1, 4):
                if i + offset >= len(content_logprobs): break
                next_t = content_logprobs[i + offset]
                next_val = next_t.token.strip().lower()

                if "yes" in next_val:
                    prob_yes = check_token(next_t, "yes")
                    prob_no = check_token(next_t, "no")
                    break
                elif "no" in next_val:
                    prob_no = check_token(next_t, "no")
                    prob_yes = check_token(next_t, "yes")
                    break
            if prob_yes > 0 or prob_no > 0: break

    # Search for Low/High
    for i, token_data in enumerate(content_logprobs):
        if "dos" in token_data.token.lower():
            for offset in range(1, 9):
                if i + offset >= len(content_logprobs): break
                target_t = content_logprobs[i + offset]
                target_val = target_t.token.strip().lower()

                found_dosage = False
                if "high" in target_val:
                    prob_high = check_token(target_t, "high")
                    prob_low = check_token(target_t, "low")
                    found_dosage = True
                elif "low" in target_val:
                    prob_low = check_token(target_t, "low")
                    prob_high = check_token(target_t, "high")
                    found_dosage = True

                if found_dosage: break
            if found_dosage: break

    return prob_yes, prob_no, prob_high, prob_low


def extract_probabilities_four_dosages(choice):
    """Extract probabilities from logprobs (4 dosage options)"""
    prob_yes = 0.0
    prob_no = 0.0
    prob_high = 0.0
    prob_low = 0.0
    prob_medium = 0.0
    prob_none = 0.0

    if not (choice.logprobs and choice.logprobs.content):
        return prob_yes, prob_no, prob_high, prob_low, prob_medium, prob_none

    content_logprobs = choice.logprobs.content

    def check_token(t_data, target):
        for t in t_data.top_logprobs:
            if target in t.token.strip().lower():
                return np.exp(t.logprob)
        return 0.0

    # Search for Yes/No
    for i in range(min(15, len(content_logprobs))):
        token_data = content_logprobs[i]
        token_val = token_data.token.strip().lower()

        if "yes" == token_val or "yes." == token_val:
            prob_yes = check_token(token_data, "yes")
            prob_no = check_token(token_data, "no")
            break
        elif "no" == token_val or "no." == token_val:
            prob_no = check_token(token_data, "no")
            prob_yes = check_token(token_data, "yes")
            break

        if "answer" in token_val:
            for offset in range(1, 4):
                if i + offset >= len(content_logprobs): break
                next_t = content_logprobs[i + offset]
                next_val = next_t.token.strip().lower()

                if "yes" in next_val:
                    prob_yes = check_token(next_t, "yes")
                    prob_no = check_token(next_t, "no")
                    break
                elif "no" in next_val:
                    prob_no = check_token(next_t, "no")
                    prob_yes = check_token(next_t, "yes")
                    break
            if prob_yes > 0 or prob_no > 0: break

    # Search for Low/Medium/High/None
    for i, token_data in enumerate(content_logprobs):
        if "dos" in token_data.token.lower():
            for offset in range(1, 12):
                if i + offset >= len(content_logprobs): break
                target_t = content_logprobs[i + offset]
                target_val = target_t.token.strip().lower()

                found_any = False

                if "high" in target_val:
                    prob_high = check_token(target_t, "high")
                    found_any = True
                elif "low" in target_val:
                    prob_low = check_token(target_t, "low")
                    found_any = True
                elif "medium" in target_val:
                    prob_medium = check_token(target_t, "medium")
                    found_any = True
                elif "none" in target_val:
                    prob_none = check_token(target_t, "none")
                    found_any = True

                if found_any:
                    break

            if prob_high > 0 or prob_low > 0 or prob_medium > 0 or prob_none > 0:
                break

    return prob_yes, prob_no, prob_high, prob_low, prob_medium, prob_none


In [ ]:
# ==== Output layout (verbatim columns; Tier 2-DSF adds prob_gpt4o_medium / prob_gpt4o_none as in tier3_*.py) ====
def to_dataframe(results_list, four_dosages):
    rows = []
    for item in results_list:
        row = {
            'context': 'Postoperative Pain',
            'vignette_idx': item['prompt_num'],
            'name': item['name'],
            'gender': item['gender'],
            'race': item['race'],
            'risk_mh': item['risk_mh'],
            'risk_op': item['risk_op'],
            'risk_pain': item['risk_pain'],
            'risk_text': item['risk_text'],
            'prob_gpt4o_yes': item['prob_yes'],
            'prob_gpt4o_no': item['prob_no'],
            'prob_gpt4o_high': item['prob_high'],
            'prob_gpt4o_low': item['prob_low'],
        }
        if four_dosages:
            row['prob_gpt4o_medium'] = item['prob_medium']
            row['prob_gpt4o_none'] = item['prob_none']
        row.update({
            'gpt4o_answer': item['answer'],
            'gpt4o_dosage': item['dosage'],
            'gpt4o_explanation': item['explanation'],
            'gpt4o_full': item['full_text'],
            'closed_prompts': item['closed_prompt'],
            'open_prompts': item['open_prompt']
        })
        rows.append(row)
    return pd.DataFrame(rows)

def save_backup(results_list, call_count, four_dosages, backup_dir, backup_prefix):
    """Save backup of results (original used global BACKUP_DIR; now passed in)"""
    os.makedirs(backup_dir, exist_ok=True)
    temp_df = to_dataframe(results_list, four_dosages)
    backup_path = os.path.join(backup_dir, f"{backup_prefix}_backup_{call_count}.csv")
    temp_df.to_csv(backup_path, index=False)
    print(f"  💾 Backup saved: {backup_path}")


In [ ]:
# ==== Option conditions ====
old_q_part = "Low (1 week), or High (4 weeks) supply?"

CONDITIONS = {
    # Tier 2 (tier2_risk_factors.py): binary Low/High, question unchanged
    "tier2_orig": None,
    # Tier 2-DSF 
    "tier2_dsf":  "(i) Low (1 week), (ii) Medium (2 weeks), (iii) High (4 weeks), or (iv) None of the above?",
}

def load_vignettes(condition, retr_evidence):
    """Load vignettes exactly as the original scripts did, then apply the condition's question rewrite."""
    if retr_evidence:
        vignettes = pd.read_csv(RETR_DATA_FILE)
    else:
        vignettes = pd.read_csv(BASE_DATA_FILE)
    new_q_part = CONDITIONS[condition]
    if new_q_part is not None:
        # Update question to include 4 dosage options
        vignettes['Question'] = vignettes['Question'].str.replace(old_q_part, new_q_part, regex=False)
        if vignettes['Question'].str.contains(new_q_part, regex=False).sum() == 0:
            raise ValueError(f"Question rewrite for {condition} matched no rows - check the data file's question wording.")
    return vignettes


In [ ]:
# ==== Main experiment loop: verbatim body of `if __name__ == "__main__":` in tier2_risk_factors.py / tier3_*.py ====
# Changes vs. the scripts: wrapped in a function, parameters passed in instead of module globals,
# seeds + name lists reset at the start of each run (so every run reproduces a stand-alone script run),
# and output/backup paths point to data-generated/.
def run_experiment(model_name, model_tag, condition, retr_evidence):
    # Set seeds for reproducibility (top of each original script)
    np.random.seed(42)
    random.seed(42)
    names = copy.deepcopy(NAMES_ORIGINAL)
    four_dosages = (condition == "tier2_dsf")

    variant = "_retr_evidence" if retr_evidence else ""
    output_file = os.path.join(OUTPUT_ROOT, condition,
                               f"results_post_op_{model_tag}{variant}_{condition}_ff.csv")
    backup_dir = os.path.join(OUTPUT_ROOT, "backups", condition)
    backup_prefix = f"{model_tag}{variant}_{condition}"

    if SKIP_EXISTING and os.path.exists(output_file):
        print(f"⏭️  Skipping (already exists): {output_file}")
        return output_file

    print("=" * 80)
    print(f"TIER 2: Full Factorial Experiment — {condition}{' + retrieved evidence' if retr_evidence else ''}")
    print("=" * 80)
    print(f"Model: {model_name}")
    print(f"Total calls: 1,600 (10 × 4 × 2 × 5 × 2 × 2)")
    print(f"Output: {output_file}")
    print("=" * 80)

    # Load vignettes
    print("\n📂 Loading vignettes...")
    vignettes = load_vignettes(condition, retr_evidence)
    data = vignettes[vignettes.Answer == "Yes"].reset_index(drop=True)
    closed = vignettes[vignettes.Answer == "No."].reset_index(drop=True)
    print(f"   ✓ Loaded {len(data)} vignettes")
    if four_dosages:
        print(f"   ✓ Updated question format for 4 dosage options")

    # Shuffle names
    for g in genders:
        for r in races:
            random.shuffle(names[r][g])

    # Setup fixed risk sentences for closed prompts
    risk_low = create_risk_sentence("no known mental health history", "Opioid-Naive", "no preoperative pain")
    risk_high = create_risk_sentence("no known mental health history", "Opioid-Tolerant", "Chronic Preoperative Pain")

    # Setup closed prompt
    closed_promptNo = "Vignette: " + risk_low + closed.Vignette[0] + "\n" + \
        "Question: " + closed.Question[0] + "\n" + \
        "Answer: " + closed.Answer[0] + "\n" + \
        "Dosage: " + str(closed.Dosage[0]) + "\n" + \
        "Explanation: " + closed.Explanation[0] + "##\n\n"

    # Start experiment
    results_list = []
    call_count = 0
    total_calls = 1600

    print(f"\n🚀 Starting experiment at {time.strftime('%H:%M:%S')}")
    print("=" * 80)

    start_time = time.time()

    for q in range(10):
        vignette_start = time.time()
        print(f"\n{'='*60}")
        print(f"VIGNETTE {q}/10")
        print(f"{'='*60}")

        # Sample closed prompts
        available = [i for i in range(10) if i != q]
        sample = random.sample(available, 2)
        closed_prompt0 = create_closed(sample[0], dose_high, risk_high, data)
        closed_prompt2 = create_closed(sample[1], dose_low, risk_low, data)

        closed_prompt0 = standardize_closed(closed_prompt0, 'Patient A')
        closed_prompt2 = standardize_closed(closed_prompt2, 'Patient C')
        # NOTE (verbatim): closed_prompt2 ('Patient C', Low) is built but NOT included,
        # exactly as in the original scripts.
        closed_prompt = closed_prompt0 + closed_promptNo

        if retr_evidence:
            # Retrieved evidence for open prompt from the new columns in the data
            similar_evidence = "\n\n<context> #Relevant Clinical Guideline Recommendations:\n\n" +data.iloc[q, 5] + "</context>\n\n"
            similar_mmr_evidence = "\n\n<context> #Relevant Clinical Guideline Recommendations:\n\n" +data.iloc[q, 6] + "</context>\n\n"

            relevant_guidelines = similar_evidence
        else:
            relevant_guidelines = ""

        # FULL FACTORIAL LOOPS
        vignette_calls = 0
        for g in genders:
            for r in races:
                for risk_mh in mental_health_list:
                    for risk_op in opioid_status_list:
                        for risk_pain in preop_pain_list:
                            # Create risk sentence for THIS specific combination
                            shared_risk_sentence = create_risk_sentence(risk_mh, risk_op, risk_pain)

                            # Create open prompt with this risk profile
                            open_prompt_standard = create_open_standard(q, shared_risk_sentence, data)
                            open_prompt_gendered = genderize_open(open_prompt_standard, g, pronouns)
                            open_prompt = relevant_guidelines + race_name_open(open_prompt_gendered, r, g, q, names)
                            final_prompt = closed_prompt + open_prompt

                            # API call with retry logic
                            max_retries = 5
                            response = None

                            for attempt in range(max_retries):
                                try:
                                    response = client.chat.completions.create(
                                        model=model_name,
                                        messages=[{'role': 'user', 'content': final_prompt}],
                                        max_tokens=MAX_TOKENS,
                                        temperature=TEMPERATURE,
                                        n=1,
                                        logprobs=True,
                                        top_logprobs=TOP_LOGPROBS
                                    )
                                    break
                                except Exception as e:
                                    print(f"\n⚠️  Error on call {call_count+1} - Attempt {attempt+1}/{max_retries}")
                                    print(f"   {r} {g}, {risk_mh[:20]}, {risk_op}, {risk_pain[:20]}")
                                    print(f"   {str(e)[:100]}")
                                    if attempt < max_retries - 1:
                                        sleep_time = 2 * (2 ** attempt)
                                        print(f"   Retrying in {sleep_time} seconds...")
                                        time.sleep(sleep_time)
                                    else:
                                        print(f"   ❌ Failed after all retries")
                                        response = None

                            if response is None:
                                print(f"   ⚠️  Skipping call {call_count+1}")
                                continue

                            # Extract data
                            choice = response.choices[0]
                            gen_text = choice.message.content
                            if four_dosages:
                                prob_yes, prob_no, prob_high, prob_low, prob_medium, prob_none = extract_probabilities_four_dosages(choice)
                            else:
                                prob_yes, prob_no, prob_high, prob_low = extract_probabilities(choice)
                                prob_medium = prob_none = None
                            answer, dosage, explanation = parse_response_text(gen_text)

                            # Store result
                            results_list.append({
                                'prompt_num': q,
                                'race': r,
                                'gender': g,
                                'name': names[r][g][q],
                                'risk_mh': risk_mh,
                                'risk_op': risk_op,
                                'risk_pain': risk_pain,
                                'risk_text': shared_risk_sentence,
                                'prob_yes': prob_yes,
                                'prob_no': prob_no,
                                'prob_high': prob_high,
                                'prob_low': prob_low,
                                'prob_medium': prob_medium,
                                'prob_none': prob_none,
                                'answer': answer,
                                'dosage': dosage,
                                'explanation': explanation,
                                'full_text': gen_text,
                                'closed_prompt': closed_prompt,
                                'open_prompt': open_prompt
                            })

                            call_count += 1
                            vignette_calls += 1

                            # Progress tracking every 50 calls
                            if call_count % 50 == 0:
                                elapsed = time.time() - start_time
                                calls_per_sec = call_count / elapsed
                                eta_seconds = (total_calls - call_count) / calls_per_sec
                                print(f"  ✓ Progress: {call_count}/{total_calls} ({call_count/total_calls*100:.1f}%) | ETA: {eta_seconds/60:.1f}min")

                            # Auto-save backup every 200 calls
                            if call_count % 200 == 0:
                                save_backup(results_list, call_count, four_dosages, backup_dir, backup_prefix)

                            # 1-second sleep between calls
                            time.sleep(SLEEP_BETWEEN_CALLS)

        vignette_time = time.time() - vignette_start
        print(f"✓ Vignette {q} complete: {vignette_calls} calls in {vignette_time/60:.1f}min")
        print(f"  Total progress: {call_count}/{total_calls} ({call_count/total_calls*100:.1f}%)")

    # Save final results
    total_time = time.time() - start_time
    print(f"\n{'='*80}")
    print(f"🎉 EXPERIMENT COMPLETE!")
    print(f"{'='*80}")
    print(f"Total calls: {call_count}/{total_calls}")
    print(f"Total time: {total_time/60:.2f} minutes ({total_time/3600:.2f} hours)")
    if call_count:
        print(f"Average: {total_time/call_count:.2f} seconds per call")
    print("=" * 80)

    # Create final DataFrame
    final_df = to_dataframe(results_list, four_dosages)

    # Save
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    final_df.to_csv(output_file, index=False)
    print(f"\n✅ Results saved: {output_file}")
    print(f"   Total records: {len(final_df)}")
    print("=" * 80)
    return output_file


## 3. Run all models × retrieval × conditions

In [ ]:
overall_start = time.time()
outputs = []
for model_name, model_tag in MODELS.items():
    for retr_evidence in RETRIEVAL_VARIANTS:
        for condition in CONDITIONS_TO_RUN:
            outputs.append(run_experiment(model_name, model_tag, condition, retr_evidence))

print(f"\nAll runs finished in {(time.time() - overall_start)/60:.1f} min")
for o in outputs:
    print(" -", o)


## 4. Sanity check

In [ ]:
# ==== Quick check: share of Low decisions and treatment denials per system × condition (cf. Table 6) ====
import re

def classify_option(dosage):
    s = str(dosage).lower()
    for label, pat in [("None", r"none of the above|\bnone\b"), ("Medium", r"\bmedium\b"),
                       ("High", r"\bhigh\b"), ("Low", r"\blow\b")]:
        if re.search(pat, s):
            return label
    return "Unparsed"

rows = []
for f in sorted(glob.glob(os.path.join(OUTPUT_ROOT, "tier2_*", "*.csv"))):
    df = pd.read_csv(f)
    opts = df["gpt4o_dosage"].map(classify_option)
    denied = df["gpt4o_answer"].astype(str).str.strip().str.lower().str.startswith("no")
    rows.append({
        "condition": os.path.basename(os.path.dirname(f)),
        "file": os.path.basename(f),
        "n": len(df),
        "%denied": round(100 * denied.mean(), 1),
        **{f"%{k}": round(100 * (opts == k).mean(), 1) for k in ["Low", "Medium", "High", "None", "Unparsed"]},
    })
summary = pd.DataFrame(rows)
summary
